# Latency Engine: Plug-and-Play Integration Guide

This notebook demonstrates how to load, configure, and plug the `latency-engine` metrics aggregation library into your Python applications.

## 1. Importing the Pluggable Components
We can import `LatencyHandler`, `LatencyEngineConfig`, and `load_config` directly from the package:

In [ ]:
import sys
from pathlib import Path

# Add package to path for local execution
package_dir = Path.cwd().parents[5] / 'packages' / 'python' / 'latency-engine' / 'src'
sys.path.insert(0, str(package_dir))

from latency_handler import LatencyHandler
from config import load_config
print("Imports succeeded!")

## 2. Initializing the Handler with Mock Redis
Let's create a simulated Redis client to observe how metrics are written:

In [ ]:
class MockRedis:
    def __init__(self):
        self.data = {}
        self.lists = {}
        self.hashes = {}
        self.ttls = {}

    def get(self, key):
        return self.data.get(key)

    def set(self, key, value):
        self.data[key] = value.encode('utf-8') if isinstance(value, str) else value
        return True

    def lpush(self, key, value):
        self.lists.setdefault(key, []).insert(0, value)

    def ltrim(self, key, start, end):
        if key in self.lists:
            self.lists[key] = self.lists[key][start:end+1]

    def hset(self, key, mapping):
        self.hashes.setdefault(key, {}).update(mapping)

    def hincrbyfloat(self, key, field, amount):
        h = self.hashes.setdefault(key, {})
        val = float(h.get(field, "0.0")) + amount
        h[field] = str(val)

    def incr(self, key):
        h = self.hashes.setdefault("incr_counters", {})
        val = int(h.get(key, "0")) + 1
        h[key] = str(val)

    def expire(self, key, seconds):
        self.ttls[key] = seconds

    def pipeline(self):
        return self

    def execute(self):
        return []

redis_client = MockRedis()
handler = LatencyHandler(redis_client, slo_config_path=str(package_dir / 'slo_config.yaml'))
print("LatencyHandler successfully initialized and plugged!")

## 3. Feeding Spans to the Handler
We can feed span payloads directly to the handler:

In [ ]:
mock_spans = [
    {
        "span_id": "span-abc-123",
        "model": "gpt-4",
        "endpoint": "/v1/chat/completions",
        "latency_ms_ttft": 120.0,
        "latency_ms_total": 850.0,
        "completion_tokens": 25,
        "finish_reason": "stop",
        "timestamp_utc": "2026-06-17T14:00:00Z",
        "attributes": {
            "net.dns.latency_ms": 5.2,
            "net.tcp.latency_ms": 10.5,
            "llm.queue.latency_ms": 25.0,
            "llm.inference.latency_ms": 800.0
        }
    }
]

handler.handle_spans(mock_spans)
print("Processed mock spans successfully!")

## 4. Querying Metrics in Redis
Let's verify that the sketches, TPOT values, and latency attributions were stored correctly in Redis:

In [ ]:
print("--- DDSketch Keys Stored in Redis ---")
for key in redis_client.data.keys():
    print(f"Key: {key}")

print("\n--- TPOT Metrics ---")
tpot_key = "tpot:latest:gpt-4"
if tpot_key in redis_client.lists:
    print(f"Rolling TPOTs for gpt-4: {redis_client.lists[tpot_key]}")

print("\n--- Latency Attributions ---")
attr_key = "attribution:span-abc-123"
if attr_key in redis_client.hashes:
    print(f"Span latency attribution: {redis_client.hashes[attr_key]}")
